In [1]:
# Comprobamos rutas y hacemos un recuento simple de imágenes y anotaciones

from pathlib import Path

DATA_ROOT = Path(r"C:\Users\wail-\Desktop\armas_blancas\dataset\OPIXray")

def dirs(split):
    base = DATA_ROOT / split
    img = base / f"{split}_image"
    ann = base / f"{split}_annotation"
    return img, ann

for split in ["train", "test"]:
    img_dir, ann_dir = dirs(split)
    assert img_dir.exists(), f"No existe {img_dir}"
    assert ann_dir.exists(), f"No existe {ann_dir}"
    n_img = len([p for p in img_dir.iterdir() if p.is_file()])
    n_ann = len([p for p in ann_dir.iterdir() if p.is_file()])
    print(f"{split.upper()} | imágenes: {n_img} | anotaciones: {n_ann}")

# Listamos algunas extensiones de anotación e imagen para hacernos una idea
for split in ["train", "test"]:
    img_dir, ann_dir = dirs(split)
    img_exts = {p.suffix.lower() for p in img_dir.iterdir() if p.is_file()}
    ann_exts = {p.suffix.lower() for p in ann_dir.iterdir() if p.is_file()}
    print(f"{split.upper()} | ext imágenes: {sorted(img_exts)} | ext anotaciones: {sorted(ann_exts)}")


TRAIN | imágenes: 7109 | anotaciones: 7109
TEST | imágenes: 1776 | anotaciones: 1776
TRAIN | ext imágenes: ['.jpg'] | ext anotaciones: ['.txt']
TEST | ext imágenes: ['.jpg'] | ext anotaciones: ['.txt']


In [3]:
# Rutas y conteo básico
from pathlib import Path

DATA_ROOT = Path(r"C:\Users\wail-\Desktop\armas_blancas\dataset\OPIXray")  # cambia si lo necesitas
assert (DATA_ROOT / "train" / "train_image").exists(), "No encuentro train_image"
assert (DATA_ROOT / "train" / "train_annotation").exists(), "No encuentro train_annotation"
assert (DATA_ROOT / "test"  / "test_image").exists(),  "No encuentro test_image"
assert (DATA_ROOT / "test"  / "test_annotation").exists(), "No encuentro test_annotation"

n_train_img = len(list((DATA_ROOT/"train"/"train_image").glob("*.jpg")))
n_train_lab = len(list((DATA_ROOT/"train"/"train_annotation").glob("*.txt")))
n_test_img  = len(list((DATA_ROOT/"test"/"test_image").glob("*.jpg")))
n_test_lab  = len(list((DATA_ROOT/"test"/"test_annotation").glob("*.txt")))

print("TRAIN imágenes:", n_train_img, "| anotaciones:", n_train_lab)
print("TEST  imágenes:", n_test_img,  "| anotaciones:", n_test_lab)


TRAIN imágenes: 7109 | anotaciones: 7109
TEST  imágenes: 1776 | anotaciones: 1776


In [4]:
# Conversión simple al formato YOLO y split 70 15 15
from pathlib import Path
from PIL import Image
import random, shutil

DATA_ROOT = Path(r"C:\Users\wail-\Desktop\armas_blancas\dataset\OPIXray")
OUT_ROOT  = Path(r"C:\Users\wail-\Desktop\armas_blancas\yolo_data")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# clases finales
names = ["knife", "scissors"]
name_to_id = {n:i for i,n in enumerate(names)}

# mapeo de clases originales → clases finales
class_map = {
    "Folding_Knife": "knife",
    "Straight_Knife": "knife",
    "Utility_Knife": "knife",
    "Multi-tool_Knife": "knife",
    "Scissor": "scissors",
}

def parse_lines(txt_path):
    # cada línea: filename class xmin ymin xmax ymax
    items = []
    for ln in txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines():
        parts = ln.split()
        if len(parts) != 6:
            continue
        fname, cls, xmin, ymin, xmax, ymax = parts
        if cls not in class_map:
            continue
        xmin, ymin, xmax, ymax = map(int, [xmin, ymin, xmax, ymax])
        items.append((fname, class_map[cls], xmin, ymin, xmax, ymax))
    return items

def to_yolo(img_w, img_h, xmin, ymin, xmax, ymax):
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    return cx, cy, bw, bh

def ensure_dirs(split):
    (OUT_ROOT/"images"/split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT/"labels"/split).mkdir(parents=True, exist_ok=True)

def write_example(split, img_path, yolo_rows):
    dst_img = OUT_ROOT/"images"/split/img_path.name
    dst_lab = OUT_ROOT/"labels"/split/(img_path.stem + ".txt")
    shutil.copy2(img_path, dst_img)
    with open(dst_lab, "w", encoding="utf-8") as f:
        for cls_name, cx, cy, bw, bh in yolo_rows:
            f.write(f"{name_to_id[cls_name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

# leemos todos los pares imagen-anotación de train
train_img_dir = DATA_ROOT / "train" / "train_image"
train_lab_dir = DATA_ROOT / "train" / "train_annotation"
pairs = []  # cada elemento: (img_path, yolo_rows)

for txt in train_lab_dir.glob("*.txt"):
    items = parse_lines(txt)
    if not items:
        continue
    # el archivo de imagen puede venir por nombre en la primera columna
    fname = items[0][0]
    img_path = train_img_dir / fname
    if not img_path.exists():
        # intentamos por mismo nombre que el txt
        img_path = train_img_dir / f"{txt.stem}.jpg"
        if not img_path.exists():
            continue
    with Image.open(img_path) as im:
        w, h = im.size
    yolo_rows = []
    for _, cls_name, xmin, ymin, xmax, ymax in items:
        cx, cy, bw, bh = to_yolo(w, h, xmin, ymin, xmax, ymax)
        yolo_rows.append((cls_name, cx, cy, bw, bh))
    if yolo_rows:
        pairs.append((img_path, yolo_rows))

# barajamos y dividimos 85% train y 15% val
random.seed(42)
random.shuffle(pairs)
cut = int(len(pairs) * 0.85)
train_pairs = pairs[:cut]
val_pairs   = pairs[cut:]

# escribimos
ensure_dirs("train")
ensure_dirs("val")
for img_path, rows in train_pairs:
    write_example("train", img_path, rows)
for img_path, rows in val_pairs:
    write_example("val", img_path, rows)

# ahora test
test_img_dir = DATA_ROOT / "test" / "test_image"
test_lab_dir = DATA_ROOT / "test" / "test_annotation"
test_pairs = []

for txt in test_lab_dir.glob("*.txt"):
    items = parse_lines(txt)
    if not items:
        continue
    fname = items[0][0]
    img_path = test_img_dir / fname
    if not img_path.exists():
        img_path = test_img_dir / f"{txt.stem}.jpg"
        if not img_path.exists():
            continue
    with Image.open(img_path) as im:
        w, h = im.size
    yolo_rows = []
    for _, cls_name, xmin, ymin, xmax, ymax in items:
        cx, cy, bw, bh = to_yolo(w, h, xmin, ymin, xmax, ymax)
        yolo_rows.append((cls_name, cx, cy, bw, bh))
    if yolo_rows:
        test_pairs.append((img_path, yolo_rows))

ensure_dirs("test")
for img_path, rows in test_pairs:
    write_example("test", img_path, rows)

print("Listo en:", OUT_ROOT)
print("Train imágenes:", len(train_pairs))
print("Val   imágenes:", len(val_pairs))
print("Test  imágenes:", len(test_pairs))
print("Clases:", names)


Listo en: C:\Users\wail-\Desktop\armas_blancas\yolo_data
Train imágenes: 6042
Val   imágenes: 1067
Test  imágenes: 1776
Clases: ['knife', 'scissors']


In [5]:
# Escribimos el dataset.yaml para entrenar con YOLO
from pathlib import Path
import yaml

OUT_ROOT = Path(r"C:\Users\wail-\Desktop\armas_blancas\yolo_data")

cfg = {
    "path": str(OUT_ROOT),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "names": ["knife", "scissors"]
}

with open(OUT_ROOT/"dataset.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Escrito:", OUT_ROOT/"dataset.yaml")
print((OUT_ROOT/"dataset.yaml").read_text(encoding="utf-8"))


Escrito: C:\Users\wail-\Desktop\armas_blancas\yolo_data\dataset.yaml
path: C:\Users\wail-\Desktop\armas_blancas\yolo_data
train: images/train
val: images/val
test: images/test
names:
- knife
- scissors



In [6]:
# smoke test con YOLOv8s
# si no tienes ultralytics, descomenta la línea de instalación
# !pip install -q ultralytics

from pathlib import Path
import torch
from ultralytics import YOLO

DATASET_YAML = r"C:\Users\wail-\Desktop\armas_blancas\yolo_data\dataset.yaml"
assert Path(DATASET_YAML).exists(), "No encuentro dataset.yaml. Ejecuta antes la conversión y creación de carpetas."

# elegimos dispositivo
device = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", "GPU" if device == 0 else "CPU")

# cargamos pesos base y lanzamos entrenamiento corto
model = YOLO("yolov8s.pt")
results = model.train(
    data=DATASET_YAML,
    epochs=3,         # pocas épocas para probar que todo funciona
    imgsz=640,
    batch=16,
    device=device,
    project=r"C:\Users\wail-\Desktop\armas_blancas\runs",
    name="smoke_test",
    exist_ok=True,
    verbose=True
)

print("Entrenamiento de prueba terminado.")
print("Revisa la carpeta runs para ver logs, gráficos y el mejor checkpoint.")


Dispositivo: GPU
New https://pypi.org/project/ultralytics/8.3.229 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.223  Python-3.10.18 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\wail-\Desktop\armas_blancas\yolo_data\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic